# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed. If not present, install it.
!pip install mlcroissant pandas matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print metadata summary
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

The Croissant schema organizes data in record sets. Let's discover their IDs and preview available fields for each.

In [ ]:
# List recordSet @ids
record_sets = dataset.record_sets

print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', '<no name>')}" )

# For each record set, list fields and columns by their @id
for rs in record_sets:
    print(f"\nRecord set @id: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field']
        if not isinstance(fields, list):
            fields = [fields]
        print("Fields:")
        for f in fields:
            print(f"  - @id: {f['@id']}")
            if 'column' in f:
                columns = f['column']
                if not isinstance(columns, list):
                    columns = [columns]
                print("    Columns:")
                for c in columns:
                    print(f"      - @id: {c['@id']} (name: {c.get('name','<no name>')})")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview above.

In [ ]:
# Select record sets by their @id
# Let's use the record sets discovered above
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)

# Show columns from the first loaded record set
if len(dataframes) > 0:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"Loaded DataFrame columns for record set @id: {first_record_set_id}")
    print(dataframes[first_record_set_id].columns.tolist())
    dataframes[first_record_set_id].head()
else:
    print("No dataframes were created. Please check record set IDs.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing numeric fields, and grouping data by key attributes using column `@id`s.

In [ ]:
# For demonstration, select a numeric field and perform filtering and normalization.
# Use @id for columns. Replace these with known column @id values from the overview.

if len(dataframes) > 0:
    df = dataframes[first_record_set_id]

    # Try to find numeric columns for analysis
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()

    if numeric_columns:
        numeric_field_id = numeric_columns[0]

        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by a categorical field if available
        cat_columns = df.select_dtypes(include=['object']).columns.tolist()
        if cat_columns:
            group_field_id = cat_columns[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric columns found for EDA.")
else:
    print("No dataframe found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will plot the normalized numeric field and compare it across groups using their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_columns:
    # Plot the distribution of the normalized numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[norm_col], kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution of normalized {numeric_field_id}")
    plt.xlabel(norm_col)
    plt.ylabel("Frequency")
    plt.show()

    # Grouped barplot by the selected categorical field
    if cat_columns:
        plt.figure(figsize=(10,6))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=filtered_df, ci='sd')
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the Croissant-defined dataset and identified its record sets, fields, and columns by their unique `@id`.
- Demonstrated extraction and basic EDA on the most populous record set, filtering and normalizing numeric data, and observing variations across categorical groups.
- Visualized data distributions to provide insight into adoption predictors in rangeland management, utilizing data structure from their Croissant schema.

For more advanced analysis, consult [mlcroissant documentation](https://mlcroissant.org) and the dataset's Croissant schema for additional data contexts.